In [129]:
from sklearn.linear_model import LinearRegression
from bokeh.models import HoverTool 
from bokeh.io import export_svg
import numpy as np
import transforms3d as td
import pandas as pd
import bokeh.io
import bokeh.plotting
# Import libraries
from mpl_toolkits import mplot3d
import matplotlib.pyplot as plt
import re
import panel as pn
from bokeh.models import ColumnDataSource
from bokeh.palettes import Viridis
bokeh.io.output_notebook()
pn.extension()
import plotly.graph_objects as go
from bokeh.io import export_png
import holoviews as hv
from bokeh.models import CategoricalColorMapper, Legend

Loading BokehJS ...

In [130]:
p1_opens = {'passive_start' : [[66, 110], [199, 236], [305, 348], [425, 462], [535, 578], [658, 702], [769, 810], [875, 920], [994, 1030], [1090, 1135], [1204, 1249]],
'active_rorcr' : [[1856, 1934], [2505, 2575], [3244, 3305]],
'active_cube' : [[626, 680], [1003, 1053], [1946, 1990], [3327, 3371], [4834, 4878], [5978, 6028], [6178, 6219], [6549, 6590], [6853, 6892], [7260, 7303], [7550, 7590], [9115, 9161], [9322, 9366], [9788, 9831], [10697, 10746], [11063, 11108], [16324, 16369], [17175, 17220], [17741, 17782], [18209, 18249], [18661, 18699], [23887, 23910], [24319, 24342], [24440, 24504], [26049, 26097], [26274, 26304]],
'passive_end' : [[225, 266], [319, 351], [414, 455], [501, 541], [593, 632], [705, 743], [805, 844], [896, 936], [995, 1044], [1099, 1139]]}

p13_opens = {'passive_start' : [[334, 383], [555, 600], [737, 787], [897, 933], [1031, 1075], [1166, 1211], [1303, 1343], [1437, 1477], [1558, 1598], [1683, 1730]],
             'active_rorcr' : [[2694,2941],[3872,3921],[4662,4956],[6828,6895]],
             'active_cube' : [[3483, 3534], [3651, 3729], [4757, 4816], [4871, 4893], [5470, 5498], [6234, 6291], [6669, 6732], [7009, 7053], [7457, 7505], [8018, 8049], [8190, 8233], [8458, 8503], [8668, 8740], [9261, 9296], [9351, 9419], [9584, 9626], [10539, 10570], [10613, 10649], [10770, 10822], [11033, 11072], [11383, 11423], [11692, 11732], [12106, 12144], [12716, 12766], [12945, 12990], [13125, 13157], [13230, 13269], [13873, 13918], [14012, 14061], [15229, 15274]],
             'passive_end' : [[129,164],[265,356],[438,518],[665,701],[843,881],[963,1050],[1186,1225],[1334,1372],[1506,1536],[1647,1687]]}

p3_old_opens = {'passive_start' : [[316, 371], [810, 866], [1029, 1080], [1228, 1282], [1427, 1476], [1623, 1679], [1805, 1864], [1979, 2030], [2119, 2178], [2272, 2329]],
            'active_rorcr' : [[6454, 6523], [7650, 7753], [9392, 9444]],
            'active_cube' : [[2328, 2696], [4601, 4651], [5807, 5861], [14749, 14812], [15108, 15186]],
            'passive_end' : [[146, 198], [552, 610], [858, 913], [1099, 1148], [1273, 1327], [1486, 1537], [1681, 1736], [1885, 1936], [2055, 2111], [2253, 2303]]}

p3_opens = {'passive_start': [[150, 196], [384, 432], [568, 606], [751, 794], [928, 963], [1097, 1142], [1271, 1311], [1442, 1483], [1592, 1631], [1751, 1804]],
            'active_rorcr': [[1876, 1992], [2709, 2753], [3484, 3531]],
            'active_cube' : [[2593, 2714], [2744, 3019], [5558, 5601], [7737, 7794], [8232, 8259], [8439, 8456], [8810, 8851], [9665, 9678], [13075, 13085], [14028, 14043], [14100, 14134], [14184, 14218], [15326, 15342], [17772, 17808], [17890, 17913], [17955, 17977], [17986, 17993], [18030, 18045], [18203, 18221], [18279, 18311], [19024, 19030], [19040, 19059], [19136, 19163], [19548, 19587], [20972, 21006], [21073, 21094], [21159, 21183], [21480, 21501], [21528, 21537], [21602, 21615]],
            'passive_end' : [[87, 142], [263, 324], [469, 528], [662, 703], [854, 899], [1050, 1091], [1232, 1281], [1419, 1454], [1590, 1633], [1766, 1807], [1942, 1982]]}

opens = {'p1' : p1_opens, 'p13': p13_opens, 'p3' : p3_opens}
avg_best_fit = {'passive_start' : [], 'active_rorcr' : [], 'active_cube' : [], 'passive_end': []}
plot_label = {'passive_start': 'P1 Phase',
 'passive_end': 'P2 Phase',
 'active_rorcr': 'T4 Phase',
 'active_rorcr1': 'T4 Phase',
 'active_cube': 'Functional Training Phase',
 'mcp_angle': 'MCP Angle',
 'pip_angle': 'PIP Angle',
 'm': 'Stiffness (N/mm)',
 'motor_position': 'Cable Retraction (mm)',
 'futek': 'Force (N)',
 'time_open': 'Time over Experimental Condition (unscaled)',
 'p1': 'S1',
 'p13': 'S2',
 'p3': 'S3'}

In [132]:
def get_df(file):
    df = pd.read_csv(file)
    df['time_unitless'] = df['index']
    df.set_index(['condition', 'c_index', 'index'], inplace=True)
    df['open'] = 0
    df['m'] = 0
    df['b'] = 0
    df['time_open']=0

    return df

def get_opens(df, opens_dict, avg_best_fit):
    d = 1
    for c in opens_dict.keys():
        o=1
        m = 0
        b = 0
        for open in opens_dict[c]:
            i = open[0]
            f = open[1]
            df.loc[(c, np.r_[i:f], slice(None)), 'open'] = o
            x_vals = df.loc[(c, np.r_[i:f], slice(None)), 'motor_position'].copy().values.reshape(-1, 1)
            y_vals = df.loc[(c, np.r_[i:f], slice(None)), 'futek'].copy().values
            model = LinearRegression()
            model.fit(x_vals, y_vals)
            m += model.coef_
            b += model.intercept_
            df.loc[(c, np.r_[i:f], slice(None)), 'm'] = np.ones(f-i)*model.coef_
            df.loc[(c, np.r_[i:f], slice(None)), 'b'] = np.ones(f-i)*model.intercept_
            df.loc[(c, np.r_[i:f], slice(None)), 'time_open'] = d
            o += 1
            d+=1
        avg_best_fit[c] = [m/(o-1), b/(o-1)]
    df = df[df['open']>0]
    df.reset_index(inplace = True)
    df.set_index(['condition', 'open'], inplace = True)

    return df

def get_mcp_pip_plot(df, condition, plot_label):
        x = 'c_index'
        y = 'mcp_angle'
        z = 'pip_angle'
        df = df.loc[(condition, slice(None), slice(None)), :].copy().reset_index()
        hover = HoverTool(tooltips=[("index", "$index"),])
        p = bokeh.plotting.figure(
                width=800,
                height=600,
                x_axis_label="Time over Experimental Condition (unscaled)",
                y_axis_label="Angle (degres)",
                title = plot_label[condition] + ' MCP and PIP angles',
                y_range = [-45, 200],
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset']
            )

        p.circle(df[x], df[y], legend_label = plot_label[y], alpha = 0.5)
        p.circle(df[x], df[z], color = 'red', legend_label = plot_label[z], alpha = 0.5)

        return p

def get_angle_plots(df, conditions, plot_label):
    plts = []
    for i in conditions:
        p = get_mcp_pip_plot(df, i, plot_label)
        p.title.text_font_size = '20pt'
        p.legend.label_text_font_size = '20pt'
        p.xaxis.axis_label_text_font_size = '20pt'
        p.yaxis.axis_label_text_font_size = '20pt'
        p.yaxis.major_label_text_font_size = '18pt'
        p.xaxis.major_label_text_font_size = '18pt'
        #p.legend.click_policy = 'hide'
        bokeh.io.show(p)
        plts.append(p)
    return plts


In [ ]:
patient = 'p3'
file = '/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/parsed_data/%s_parsed_data.csv'%patient
conditions = opens[patient].keys()
df = get_df(file)
df3 = get_opens(df, opens[patient])

patient = 'p13'
file = '/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/parsed_data/%s_parsed_data.csv'%patient
df = get_df(file)
df2 = get_opens(df, opens[patient])


In [133]:
patient = 'p1'
file = '/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/parsed_data/%s_parsed_data.csv'%patient
df = get_df(file)
df1 = get_opens(df, opens[patient], avg_best_fit)


In [ ]:
def get_xy_plot(df_, x, y, z=False):

    title = "%s vs. %s"%(y, x)
    if z != False:
        title = "%s and %s vs. %s"%(y, z, x)
    col = bokeh.palettes.d3['Category10'][10]
    hover = HoverTool(tooltips=[("index", "$index"),])
    p = bokeh.plotting.figure(
            width=1000,
            height=600,
            x_axis_label=x,
            y_axis_label=y,
            title = title,
            tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset']
        )
    p.add_layout(Legend(), 'right')
    for i, c in enumerate(['passive_start', 'active_cube', 'active_rorcr', 'passive_end']):
        df = df_.loc[(c, slice(None), slice(None)), :].copy().reset_index()
        p.circle(df[x], 0.3*df[y], color = col[i], legend_label = c)
        if z != False: 
            p.line(df[x], df[z], color = 'grey', line_width = 3, legend_label = z)

    p.legend.location='top_left'
    p.legend.click_policy='hide'
    bokeh.io.show(p)
    return p

In [ ]:
df3 = df1.loc[('active_cube', 10)].copy()

p = bokeh.plotting.figure(
        width=1000,
        height=600
    )
p.add_layout(Legend(), 'right')
p.circle(df3['motor_position'], df3['futek'], color = 'red')

p.legend.location='top_left'
p.legend.click_policy='hide'
bokeh.io.show(p)

In [ ]:
get_xy_plot(df, 'time_unitless', 'motor_position', 'futek')

In [ ]:
plts = get_angle_plots(df2, conditions, plot_label)

In [ ]:
for i, c in enumerate(conditions):
  num_opens = df1.loc[(c, slice(None))].index.values.max()
  df1.loc[(c, slice(None)), 'motor_position']
  for o in range(1, num_opens+1):
    df2 = df1.loc[(c, o)].copy()
    

In [ ]:
plts = []
x = 'motor_position'
y = 'futek'

hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.d3['Category10'][10]
p = bokeh.plotting.figure(
        width=1000,
        height=600,
        y_axis_label=plot_label[y],
        x_axis_label=plot_label[x],
        title = '%s Force vs. Cable Retraction during Hand Opening'%plot_label[patient],
        #y_range = [-2, 20],
        tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )

p.add_layout(Legend(), 'right')

for i, c in enumerate(conditions):
  num_opens = df1.loc[(c, slice(None))].index.values.max()
  df1.loc[(c, slice(None)), 'motor_position']
  for o in range(1, num_opens+1):
    p.line(df1.loc[(c, o), x], df1.loc[(c, o), y], color = col[i], legend_label = c, line_width = 3, alpha = 0.5)

p.title.text_font_size = '20pt'
p.legend.label_text_font_size = '20pt'
p.xaxis.axis_label_text_font_size = '20pt'
p.yaxis.axis_label_text_font_size = '20pt'
p.yaxis.major_label_text_font_size = '18pt'
p.xaxis.major_label_text_font_size = '18pt'
p.legend.location='bottom'
p.legend.click_policy='hide'
plts.append(p)
bokeh.io.show(p)

In [138]:
x = 'motor_position'
y = 'futek'


hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.d3['Category10'][10]
p = bokeh.plotting.figure(
        width=1000,
        height=600,
        y_axis_label=plot_label[y],
        x_axis_label=plot_label[x],
        title = '%s Force vs. Cable Retraction during Hand Opening'%plot_label[patient],
        #y_range = [-2, 20],
        tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
p.add_layout(Legend(), 'right')

for i, c in enumerate(conditions):
  p.circle(df1.loc[(c, slice(None)), x], df1.loc[(c, slice(None)), y], color = col[i], legend_label = plot_label[c], size = 10, alpha = 0.1, line_alpha=0)
  t = np.linspace(0, 30, 100)
for i, c in enumerate(conditions):
  p.line(t, t*avg_best_fit[c][0] + avg_best_fit[c][1], color = col[i], legend_label = plot_label[c], line_width = 5)
p.title.text_font_size = '20pt'
p.legend.label_text_font_size = '20pt'
p.xaxis.axis_label_text_font_size = '20pt'
p.yaxis.axis_label_text_font_size = '20pt'
p.yaxis.major_label_text_font_size = '18pt'
p.xaxis.major_label_text_font_size = '18pt'

p.legend.click_policy='hide'
plts.append(p)
bokeh.io.show(p)

In [ ]:
x = 'motor_position'
y = 'futek'
hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.Turbo256[::-1]

for i, c in enumerate(conditions):
  num_opens = df1.loc[(c, slice(None))].index.values.max()
  p = bokeh.plotting.figure(
    width=1000,
    height=600,
    y_axis_label=plot_label[y],
    x_axis_label=plot_label[x],
    title = '%s Force vs. Cable Retraction during Hand Opening'%plot_label[patient],
    #y_range = [-2, 20],
    tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
  p.add_layout(Legend(), 'right')
  for o in range(1, num_opens+1):
    df2 = df1.loc[(c, o)].copy()
    p.line(df2[x], df2[y], color = col[::int(256/num_opens-1)][o], legend_label = str(o), line_width = 3, alpha = 0.5)

  p.title.text_font_size = '20pt'
  p.legend.label_text_font_size = '8pt'
  p.xaxis.axis_label_text_font_size = '20pt'  
  p.yaxis.axis_label_text_font_size = '20pt'
  p.yaxis.major_label_text_font_size = '18pt'
  p.xaxis.major_label_text_font_size = '18pt'
  p.legend.location='top_left'
  p.legend.click_policy='hide'
  legend = bokeh.models.Legend(location=(10,10))
  p.add_layout(legend, 'right')
  #plts.append(p)
  bokeh.io.show(p)

In [ ]:
x = 'motor_position'
y = 'futek'
hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.Turbo256[::-1]

for i, c in enumerate(conditions):
  num_opens = df1.loc[(c, slice(None))].index.values.max()
  p = bokeh.plotting.figure(
    width=1000,
    height=600,
    y_axis_label=plot_label[y],
    x_axis_label=plot_label[x],
    title = '%s Best Fit Lines during Hand Opening'%plot_label[patient],
    #y_range = [-2, 20],
    tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
  p.add_layout(Legend(), 'right')
  for o in range(1, num_opens+1):
    df2 = df1.loc[(c, o)].copy()
    t = np.linspace(0, 50, 100)
    p.line(t, df2['m'][0]*t + df2['b'][0], color = col[::int(256/num_opens-1)][o], legend_label = str(o), line_width = 3, alpha = 0.5)
    p.line(df2[x], df2[y], color = col[::int(256/num_opens-1)][o], legend_label = str(o), line_width = 3, alpha = 0.5)

  p.title.text_font_size = '20pt'
  p.legend.label_text_font_size = '8pt'
  p.xaxis.axis_label_text_font_size = '20pt'  
  p.yaxis.axis_label_text_font_size = '20pt'
  p.yaxis.major_label_text_font_size = '18pt'
  p.xaxis.major_label_text_font_size = '18pt'
  p.legend.location='top_left'
  p.legend.click_policy='hide'
  legend = bokeh.models.Legend(location=(10,10))
  p.add_layout(legend, 'right')
  #plts.append(p)
  bokeh.io.show(p)

In [ ]:
df3.reset_index(inplace = True)
df3.set_index(['condition', 'open'], inplace = True)
df3

In [ ]:
df3.loc[('passive_start', 1)]

In [ ]:
y = 'pip_angle'
x = 'mcp_angle'
hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.d3['Category10'][10]
colo = bokeh.palettes.Turbo256[::-1]
df = df1
p = bokeh.plotting.figure(
        width=1000,
        height=600,
        y_axis_label=plot_label[y],
        x_axis_label=plot_label[x],
        title = '%s MCP Angles for Hand Opening over Experimental Condition'%plot_label[patient],
        y_range = [-20, 125],
        x_range = [-20, 125],
        tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
df_ = df[df['open']>0].copy()
p.add_layout(Legend(), 'right')
for i, c in enumerate(conditions):
  dfd = df_.loc[(c, slice(None), slice(None)), :].copy().reset_index()
  p.circle(dfd[x], dfd[y], color = colo[i], legend_label = plot_label[c])
t = np.linspace(-20,120, 100)
p.line(t, t)
p.line(t, 1.4*t)
p.title.text_font_size = '20pt'
p.legend.label_text_font_size = '20pt'
p.xaxis.axis_label_text_font_size = '20pt'
p.yaxis.axis_label_text_font_size = '20pt'
p.yaxis.major_label_text_font_size = '18pt'
p.xaxis.major_label_text_font_size = '18pt'
p.legend.location='top_left'
p.legend.click_policy='hide'
plts.append(p)
bokeh.io.show(p)

In [ ]:
df['time_unitless'] = np.arange(len(df))

In [ ]:
y = 'pip_angle'
x = 'mcp_angle'
hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.d3['Category10'][10]
colo = bokeh.palettes.Turbo256[::-1]
df = df1
p = bokeh.plotting.figure(
        width=1000,
        height=600,
        y_axis_label=plot_label[y],
        x_axis_label=plot_label[x],
        title = '%s MCP Angles for Hand Opening over Experimental Condition'%plot_label[patient],
        y_range = [-20, 125],
        x_range = [-20, 125],
        tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
df_ = df[df['open']>0].copy()
p.add_layout(Legend(), 'right')
for i, c in enumerate(conditions):
  dfd = df_.loc[(c, slice(None), slice(None)), :].copy().reset_index()
  p.circle(dfd[x], dfd[y], color = colo[i], legend_label = plot_label[c])
t = np.linspace(-20,120, 100)
p.line(t, t)
p.line(t, 1.4*t)
p.title.text_font_size = '20pt'
p.legend.label_text_font_size = '20pt'
p.xaxis.axis_label_text_font_size = '20pt'
p.yaxis.axis_label_text_font_size = '20pt'
p.yaxis.major_label_text_font_size = '18pt'
p.xaxis.major_label_text_font_size = '18pt'
p.legend.location='top_left'
p.legend.click_policy='hide'
plts.append(p)
bokeh.io.show(p)

In [ ]:
y = 'pip_angle'
x = 'mcp_angle'
hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.d3['Category10'][10]
df = df2
p = bokeh.plotting.figure(
        width=1000,
        height=600,
        y_axis_label=plot_label[y],
        x_axis_label=plot_label[x],
        title = '%s MCP Angles for Hand Opening over Experimental Condition'%plot_label[patient],
        y_range = [-20, 125],
        x_range = [-20, 125],
        tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
df_ = df[df['open']>0].copy()
p.add_layout(Legend(), 'right')
for i, c in enumerate(['passive_start', 'active_rorcr1', 'active_cube', 'passive_end']):
  dfd = df_.loc[(c, slice(None), slice(None)), :].copy().reset_index()
  p.circle(dfd[x], dfd[y], color = col[i], legend_label = plot_label[c])
t = np.linspace(-20,120, 100)
p.line(t, t)
p.title.text_font_size = '20pt'
p.legend.label_text_font_size = '20pt'
p.xaxis.axis_label_text_font_size = '20pt'
p.yaxis.axis_label_text_font_size = '20pt'
p.yaxis.major_label_text_font_size = '18pt'
p.xaxis.major_label_text_font_size = '18pt'
p.legend.location='top_left'
p.legend.click_policy='hide'
plts.append(p)
bokeh.io.show(p)

In [ ]:
y = 'pip_angle'
x = 'mcp_angle'
hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.d3['Category10'][10]
df = df3
p = bokeh.plotting.figure(
        width=1000,
        height=600,
        y_axis_label=plot_label[y],
        x_axis_label=plot_label[x],
        title = '%s MCP Angles for Hand Opening over Experimental Condition'%plot_label[patient],
        y_range = [-20, 125],
        x_range = [-20, 125],
        tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
df_ = df[df['open']>0].copy()
p.add_layout(Legend(), 'right')
for i, c in enumerate(conditions):
  dfd = df_.loc[(c, slice(None), slice(None)), :].copy().reset_index()
  p.circle(dfd[x], dfd[y], color = col[i], legend_label = plot_label[c])
t = np.linspace(-20,120, 100)
p.line(t, t)

p.title.text_font_size = '20pt'
p.legend.label_text_font_size = '20pt'
p.xaxis.axis_label_text_font_size = '20pt'
p.yaxis.axis_label_text_font_size = '20pt'
p.yaxis.major_label_text_font_size = '18pt'
p.xaxis.major_label_text_font_size = '18pt'
p.legend.location='top_left'
p.legend.click_policy='hide'
plts.append(p)
bokeh.io.show(p)

In [ ]:
x = 'time_open'
y = 'pip_angle'
hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.d3['Category10'][10]
p = bokeh.plotting.figure(
        width=1000,
        height=600,
        y_axis_label=plot_label[y],
        x_axis_label=plot_label[x],
        title = '%s PIP Angles for Hand Opening over Experimental Condition'%plot_label[patient],
        y_range = [-45, 125],
        tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
df1 = df[df['open']>0].copy()
p.add_layout(Legend(), 'right')
for i, c in enumerate(conditions):
  dfd = df1.loc[(c, slice(None), slice(None)), :].copy().reset_index()
  p.circle(dfd[x], dfd[y], color = col[i], legend_label = plot_label[c])
 
p.title.text_font_size = '20pt'
p.legend.label_text_font_size = '20pt'
p.xaxis.axis_label_text_font_size = '20pt'
p.yaxis.axis_label_text_font_size = '20pt'
p.yaxis.major_label_text_font_size = '18pt'
p.xaxis.major_label_text_font_size = '18pt'
p.legend.location='bottom_right'
p.legend.click_policy='hide'
plts.append(p)
bokeh.io.show(p)

In [ ]:
i = 0 

for p in plts:
    export_png(p, filename='/home/jxu/hand_orthosis_ws/src/trakstar_ros/parsed_data/figures/%s_%s.png'%(plot_label[patient], i))
    i+=1

In [ ]:
x = 'motor_position'
y = 'futek'
hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
col = bokeh.palettes.d3['Category10'][10]

df1 = df[df['open']>0].copy()
for i, c in enumerate(['active_cube']):
  p = bokeh.plotting.figure(
        width=400,
        height=200,
        y_axis_label=y,
        x_axis_label=x,
        title = 'P13 Force vs. Tendon Displacement',
        #y_range = [-2, 20],
        tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
    )
  dfd = df1.loc[(c, slice(None), slice(None)), :].copy().reset_index()
  p.circle(dfd[x], dfd[y], color = col[i], legend_label = c)
  p.line(dfd[x], dfd[y], color = col[i], legend_label = c)
  p.legend.location='top_left'
  p.legend.click_policy='hide'
  bokeh.io.show(p)

